У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

# ЗАВДАННЯ 1

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [86]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder

from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.combine import SMOTETomek

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

In [30]:
drive.mount('/content/drive')
df = pd.read_csv('drive/MyDrive/customer_segmentation_train.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
df.head()

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


## Розділення на тренувальний / тестувальний набори

In [33]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Segmentation'])

In [34]:
input_cols = list(train_df.columns[1:-1])
target_col = 'Segmentation'

In [35]:
train_inputs = train_df[input_cols].copy()
train_target = train_df[target_col].copy()

test_inputs = test_df[input_cols].copy()
test_target = test_df[target_col].copy()

## Розділення колонок на числові / категоріальні

In [36]:
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
category_cols = train_inputs.select_dtypes(include='object').columns.tolist()

In [37]:
category_cols

['Gender',
 'Ever_Married',
 'Graduated',
 'Profession',
 'Spending_Score',
 'Var_1']

## Заповнення пропусків

### Числові колонки

In [38]:
train_inputs[numeric_cols].isna().sum()

,0
Age,0
Work_Experience,646
Family_Size,264


In [39]:
imputer_num = SimpleImputer(strategy = 'median')
imputer_num.fit(train_inputs[numeric_cols])

SimpleImputer(strategy='median')

In [40]:
train_inputs[numeric_cols] = imputer_num.transform(train_inputs[numeric_cols])
test_inputs[numeric_cols] = imputer_num.transform(test_inputs[numeric_cols])

### Категоріальні колонки

In [41]:
train_inputs[category_cols].isna().sum()

,0
Gender,0
Ever_Married,111
Graduated,59
Profession,106
Spending_Score,0
Var_1,60


In [42]:
imputer_cat = SimpleImputer(strategy = 'most_frequent')
imputer_cat.fit(train_inputs[category_cols])

SimpleImputer(strategy='most_frequent')

In [43]:
train_inputs[category_cols] = imputer_cat.transform(train_inputs[category_cols])
test_inputs[category_cols] = imputer_cat.transform(test_inputs[category_cols])

## Кодування категоріальних ознак для моделі

### Бінарні

In [44]:
gender_codes = {'Male': 1, 'Female': 0}
married_codes = {'Yes': 1,'No': 0}
graduated_codes = {'Yes': 1,'No': 0}

In [45]:
train_inputs['Gender_code'] = train_inputs['Gender'].map(gender_codes)
train_inputs['Ever_Married_code'] = train_inputs['Ever_Married'].map(married_codes)
train_inputs['Graduated_code'] = train_inputs['Graduated'].map(graduated_codes)

test_inputs['Gender_code'] = test_inputs['Gender'].map(gender_codes)
test_inputs['Ever_Married_code'] = test_inputs['Ever_Married'].map(married_codes)
test_inputs['Graduated_code'] = test_inputs['Graduated'].map(graduated_codes)

### Порядкові

In [46]:
df['Spending_Score'].value_counts()

,count
Spending_Score,
Low,4878
Average,1974
High,1216


In [47]:
df['Segmentation'].value_counts()

,count
Segmentation,
D,2268
A,1972
C,1970
B,1858


In [48]:
score_ordenc = OrdinalEncoder(categories=[['Low', 'Average', 'High']])
score_ordenc.fit(train_inputs[['Spending_Score']])

OrdinalEncoder(categories=[['Low', 'Average', 'High']])

In [49]:
train_inputs['Spending_Score_code'] = score_ordenc.transform(train_inputs[['Spending_Score']])
test_inputs['Spending_Score_code'] = score_ordenc.transform(test_inputs[['Spending_Score']])

### Номінальні

In [50]:
segm_labenc = LabelEncoder() # кодування таргету (LebelEncoder кодує за алфавітом)
segm_labenc.fit(train_target)

LabelEncoder()

In [51]:
train_target = segm_labenc.transform(train_target)
test_target = segm_labenc.transform(test_target)

In [52]:
onehotenc = OneHotEncoder(handle_unknown='ignore')
onehotenc.fit(train_inputs[['Profession', 'Var_1']])
onehotenc.categories_

[array(['Artist', 'Doctor', 'Engineer', 'Entertainment', 'Executive',
        'Healthcare', 'Homemaker', 'Lawyer', 'Marketing'], dtype=object),
 array(['Cat_1', 'Cat_2', 'Cat_3', 'Cat_4', 'Cat_5', 'Cat_6', 'Cat_7'],
       dtype=object)]

In [53]:
train_one_hot = onehotenc.transform(train_inputs[['Profession', 'Var_1']]).toarray()
test_one_hot = onehotenc.transform(test_inputs[['Profession', 'Var_1']]).toarray()

In [54]:
onehot_enc_cols = onehotenc.get_feature_names_out(['Profession', 'Var_1'])

In [55]:
train_inputs[onehot_enc_cols] = train_one_hot
test_inputs[onehot_enc_cols] = test_one_hot

### ФІНАЛЬНІ КОЛОНКИ ДЛЯ МОДЕЛІ

In [60]:
final_cols = numeric_cols + ['Gender_code', 'Ever_Married_code', 'Graduated_code', 'Spending_Score_code'] + list(onehot_enc_cols)

X_train = train_inputs[final_cols].copy()
X_test = test_inputs[final_cols].copy()
y_train = train_target
y_test = test_target

# ЗАВДАННЯ 2

**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

## Кодування категоріальних колонок під SMOTENC
Якщо сказати SMOTENC "усі onehot колонки — категоріальні", він буде обробляти кожну з них незалежно одна від одної (бо він не знає про залежність). Тобто спочатку пройдеться по ознаці "Doctor" і визначить, яке в сусідів найпопулярніше значення по ній 1/0. Потім по ознаці "Engineer" і тд.

Можна отримати Profession_Doctor=1 і Profession_Lawyer=1 одночасно — тобто ніби він і лікар, і юрист водночас. Так як для одного спостереження набирається кілька ознак професій. У реальних даних такого рядка не існує й не може існувати (людина має рівно одну професію з цього списку) - модель отримає "неможливого" клієнта і вчитиметься на шумі.

Тому робимо Profession_code / Var_1_code — одна ціла колонка на одну ознаку (наприклад, Profession_code = 3 означає "Doctor", а не 9 окремих прапорців).

Тоді SMOTENC "голосує" один раз за всю професію цілком, і штучний клієнт завжди отримує рівно одну - реальну, несуперечливу - професію.

**КОРОТКО:**
1. кодуємо - щоб модель могла рахувати
2. **SMOTENC розрізняє колонки за типом поведінки (усереднювати чи "голосувати" за найпопульрнішого)**, а не за тим, текст там чи число
3. one-hot розгортаємо тільки в самому кінці - під модель, — бо для ресемплінгу він, навпаки, шкодить (створює суперечливі рядки)

***ДОДАТОК***

Технічно останні версії imbalanced-learn можуть приймати сирий текст ("Doctor", "Lawyer") у цих колонках напряму, без попереднього кодування. Кодуємо їх заздалegідь не заради SMOTENC, а тому що так простіше й надійніше (менше залежності від версії бібліотеки).


In [66]:
# Окреме компактне (не one-hot) кодування Profession і Var_1, яке потрібне лише для SMOTENC/SMOTE-Tomek

prof_var_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
prof_var_enc.fit(train_inputs[['Profession', 'Var_1']])

train_inputs[['Profession_code', 'Var_1_code']] = prof_var_enc.transform(train_inputs[['Profession', 'Var_1']])
test_inputs[['Profession_code', 'Var_1_code']] = prof_var_enc.transform(test_inputs[['Profession', 'Var_1']])

## Індексація категоріальних колонок

***НАВІЩО?***

Коли передаємо дані в SMOTENC, даємо йому не таблицю з підписаними колонками (Age, Gender_code і тд), а, по суті, "голий" масив чисел - рядки й стовпці, пронумеровані як 0, 1, 2, 3...

**SMOTENC не знає назв колонок, він знає лише позиції**. Тому йому треба явно сказати: колонки під номерами 3, 4, 5, 6, 7, 8 — це категоріальні, решту (0, 1, 2) треба обробляти як звичайні числа.

In [67]:
resample_cat_cols = ['Gender_code', 'Ever_Married_code', 'Graduated_code', 'Spending_Score_code', 'Profession_code', 'Var_1_code']
resample_cols = numeric_cols + resample_cat_cols

cat_cols_indices = [resample_cols.index(c) for c in resample_cat_cols]
print('Індекси категоріальних ознак:', cat_cols_indices)

X_train_resample = train_inputs[resample_cols].copy()

Індекси категоріальних ознак: [3, 4, 5, 6, 7, 8]


###**Базовий SMOTE**
*лише на некатегоріальних (числових) ознаках*

`SMOTE` генерує нові приклади інтерполяцією між сусідами.

Для чисел це коректно (штучне спост. йде з усередненними ознаками: `оригінал + випадкове число × (сусід − оригінал)`, а от для категорій (навіть закодованих цілими числами) інтерполяція за таким принципом створила б неіснуючі значення категорій. Тому тут застосовуємо його лише до числових колонок, повністю відкинувши категоріальні — навмисне спрощення, яке демонструє обмеження базового методу.

In [72]:
smote = SMOTE(random_state=42)
X_train_smote_num, y_train_smote = smote.fit_resample(X_train_resample[numeric_cols], y_train)

### **SMOTENC**

`SMOTENC` розуміє, які колонки категоріальні (через `categorical_features`), і для них не інтерполює, а **бере значення від одного з найближчих сусідів**, тоді як для числових ознак **виконує звичайну інтерполяцію SMOTE**. Завдяки цьому можна коректно збалансувати класи, зберігши **всі** ознаки.

In [74]:
smote_nc = SMOTENC(categorical_features=cat_cols_indices, random_state=42)
X_train_smotenc, y_train_smotenc = smote_nc.fit_resample(X_train_resample, y_train)

### **SMOTE-Tomek (SMOTEK)**

`SMOTEK` = оверсемплінг SMOTE + очищення межових/шумних точок методом Tomek links (видаляються пари найближчих сусідів різних класів, що межують один з одним).

Всередині за замовчуванням використовується базовий `SMOTE`, тож щоб коректно врахувати категоріальні ознаки, передаємо туди наш `SMOTENC` через параметр `smote=`.

In [76]:
smote_tomek = SMOTETomek(smote=smote_nc, random_state=42)
X_train_smotek, y_train_smotek = smote_tomek.fit_resample(X_train_resample, y_train)

# ЗАВДАННЯ 3

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [93]:
def to_model_features(X_part):
  X_part = X_part.copy()

  # беремо закодовані для ресемплінгу колонки і повертаємо їм категоріальні значення назад -> кодуємо за тим самим onehot енкодером, що напочатку
  onehot_arr = onehotenc.transform(prof_var_enc.inverse_transform(X_part[['Profession_code', 'Var_1_code']])).toarray()

  # для закодованих колонок через onehot наповнюємо закодованими даними після ресемплінгу
  X_part[onehot_enc_cols] = onehot_arr

  return X_part[final_cols]

In [98]:
X_train_smotenc_model = to_model_features(X_train_smotenc)
X_train_smotek_model = to_model_features(X_train_smotek)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [102]:
def train_and_evaluate(X_tr, y_tr, label):

  log_reg = LogisticRegression(solver='liblinear')
  ovr_model = OneVsRestClassifier(log_reg)
  ovr_model.fit(X_tr, y_tr)
  ovr_predictions = ovr_model.predict(X_test)

  print(f'===== {label} =====')
  print(classification_report(y_test, ovr_predictions))

  return ovr_model

In [103]:
model_original = train_and_evaluate(X_train, y_train, 'Оригінальні дані (без балансування)')
model_smotenc = train_and_evaluate(X_train_smotenc_model, y_train_smotenc, 'Збалансовані SMOTENC')
model_smotek = train_and_evaluate(X_train_smotek_model, y_train_smotek, 'Збалансовані SMOTE-Tomek')

===== Оригінальні дані (без балансування) =====
              precision    recall  f1-score   support

           0       0.41      0.45      0.43       394
           1       0.42      0.15      0.22       372
           2       0.49      0.65      0.55       394
           3       0.64      0.75      0.69       454

    accuracy                           0.51      1614
   macro avg       0.49      0.50      0.47      1614
weighted avg       0.50      0.51      0.49      1614

===== Збалансовані SMOTENC =====
              precision    recall  f1-score   support

           0       0.41      0.48      0.44       394
           1       0.42      0.22      0.29       372
           2       0.51      0.61      0.55       394
           3       0.67      0.71      0.69       454

    accuracy                           0.52      1614
   macro avg       0.50      0.51      0.49      1614
weighted avg       0.51      0.52      0.50      1614

===== Збалансовані SMOTE-Tomek =====
            

## ВИСНОВОК

**Яку метрику обираємо для порівняння?**

Обираємо **`macro avg F1-score`** (усереднений F1 по всіх 4 класах з однаковою вагою). Причина: класи незбалансовані, а accuracy в такому випадку вводить в оману - модель може отримати високу accuracy, просто добре вгадуючи найбільший клас і ігноруючи менші. Macro F1 враховує якість передбачення кожного класу однаково, незалежно від його розміру — а це і є мета балансування вибірки.

**ДОДАТОК**

Мета вибору метрики — чи допомогло балансування меншим класам, а не "яка загальна частка вгаданих клієнтів" (це якраз і показують micro/accuracy — і вони "сліпі" до дисбалансу).

Weighted теж не підходить для цієї конкретної мети — вона все ще дає найбільшому класу (D) найбільшу вагу в підсумковому числі (бо зважування відбувається по розміру класу). Якщо модель ідеально вгадує D (найбільший клас) і жахливо — B (найменший), weighted avg все одно вийде "непогано", просто тому що D переважає за кількістю — і саме цю проблему ("модель ігнорує менші класи, бо вигідно просто частіше вгадувати великий") ми й намагаємось відловити.

Macro — єдина з трьох, де провал по класу B не ховається за хорошим результатом по класу D. Саме тому вона найкраще відповідає меті: "чи стала модель краще розрізняти всі сегменти, а не тільки найбільший".

**Яка модель найкраща?**

За обраною метрикою (macro F1-score):

- оригінальні дані: 0.47
- SMOTENC: 0.49
- SMOTE-Tomek: 0.49

`SMOTENC` і `SMOTE-Tomek` практично ідентичні між собою (різниця лише в `accuracy`: 0.52 проти 0.51, а `weighted avg` взагалі однаковий — 0.50), і обидва помітно кращі за модель без балансування. Отже, найкраща — будь-яка з двох збалансованих версій (`SMOTENC` чи `SMOTE-Tomek` — різниця між ними статистично несуттєва). Але як простіша без додаткового кроку очищення Tomek-links, то **`SMOTENC` ліпша**.

Показовий доказ користі балансування — клас B (він же клас 1 у звіті, найменший за розміром — 1858 прикладів): `recall` зріс з 0.15 (оригінал) до 0.22–0.23 (після ресемплінгу). Тобто без балансування модель майже завжди пропускала клієнтів класу B, а після балансування почала розпізнавати їх частіше.

**Чому різниця між SMOTENC і SMOTE-Tomek (і взагалі покращення) не дуже велика?**

Дивлячись на конкретні цифри: навіть після балансування `precision` для класу B лишився майже той самий (0.41–0.42 у всіх трьох варіантах), покращився лише `recall`. Це важлива підказка: модель не стала "розумнішою" в розрізненні B від інших класів — вона просто стала частіше ризикувати й називати клієнта класом B (менше його ігнорує), через що ловить більше правильних B, але водночас і помилок на B стає пропорційно більше — тому `precision` не зростає.

Це узгоджується з гіпотезою: сегменти A/B/C/D — бізнесове маркетингове рішення, а не природно розділювані групи за наявними ознаками (Age, Profession, Spending_Score тощо) — тобто самі класи суттєво перетинаються в просторі ознак. Логістична регресія будує лише лінійні межі, а ресемплінг вирішує проблему кількісного дисбалансу (скільки прикладів кожного класу бачить модель під час навчання), але не додає нової інформації про те, де насправді проходить межа між класами. Тому ефект від SMOTE/SMOTE-Tomek — це переважно "модель перестає ігнорувати менші класи" (вищий `recall`), а не якісний стрибок у здатності їх розрізняти (`precision` і `f1` в цілому зростають лише незначно, +0.02).